In [ ]:
import sys
from pathlib import Path
# Core libraries
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns


from imblearn.over_sampling import SMOTE
from sklearn.utils.class_weight import compute_class_weight
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
# Model saving
import os
import joblib

# Settings
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

root_path = Path.cwd().parent 
if str(root_path) not in sys.path:
    sys.path.insert(0, str(root_path))

from pipelines.data_pipeline import  split_data, get_data_and_process_target, scale_features, label_encode_target
from pipelines.Regression_pipelines import evaluate_regression_model, run_linear_baseline, run_ridge_model, run_lasso_model, run_elastic_net_model, run_poly_lasso_model, run_poly_ridge_model, run_decision_tree_suite, run_tuned_tree_model, run_random_forest_model, run_gradient_boosting_model, run_cv_leaderboard, plot_feature_importance, run_final_model_deployment, evaluate_model

print("Libraries imported successfully!")

Libraries imported successfully!


## Load Processed Data

In [2]:
# Initialize your app variables
TARGET = 'Severity'

# 1. Use the component to load data
df, target_stats = get_data_and_process_target("city_traffic_processed.csv", target_column=TARGET)

# 2. Access your range and std whenever you need them
if target_stats:
    print(f"\nReady to process models for {TARGET}...")

--- Data Successfully Loaded ---
Data shape: (500000, 114)
Target Column: 'Severity'
Target range: 3.00
Target std: 0.49

Ready to process models for Severity...


In [3]:
# Check class balance
class_counts = df[TARGET].value_counts()
class_percentages = df[TARGET].value_counts(normalize=True) * 100

print("Class Distribution:")
for cat in class_counts.index:
    print(f"{cat}: {class_counts[cat]} ({class_percentages[cat]:.1f}%)")

# Check for severe imbalance
min_class_pct = class_percentages.min()
if min_class_pct < 10:
    print(f"\nWarning: Smallest class is only {min_class_pct:.1f}% of data.")
    print("Consider adjusting your binning strategy.")
else:
    print(f"\nClass balance looks reasonable!")

Class Distribution:
2: 398335 (79.7%)
3: 84063 (16.8%)
4: 13244 (2.6%)
1: 4358 (0.9%)

Consider adjusting your binning strategy.


## Prepare Features and Target

In [4]:
X = df.drop(columns=[TARGET]).copy()
y = df[TARGET]

In [5]:
print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeatures used: {X.columns.tolist()}")
print(f"\nTarget classes: {y.unique().tolist()}")

Features shape: (500000, 113)
Target shape: (500000,)

Features used: ['Start_Lat', 'Start_Lng', 'End_Lat', 'End_Lng', 'Distance(mi)', 'Temperature(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Speed(mph)', 'Precipitation(in)', 'Amenity', 'Crossing', 'Junction', 'Station', 'Stop', 'Traffic_Signal', 'is_weekend', 'is_morning_rush', 'is_evening_rush', 'is_rush_hour', 'Geo_Cluster', 'dist_from_reg_hotspot', 'word_accident', 'word_exit', 'word_blocked', 'word_incident', 'word_lane', 'word_traffic', 'word_caution', 'word_drive', 'word_slow', 'word_closed', 'word_right', 'word_northbound', 'word_southbound', 'word_stationary', 'word_eastbound', 'word_westbound', 'word_shoulder', 'word_left', 'word_crash', 'word_delays', 'is_freezing', 'low_visibility_severity', 'has_precipitation', 'weather_cluster_clear', 'weather_cluster_cloudy', 'weather_cluster_low_visibility', 'weather_cluster_rain', 'weather_cluster_snow_ice', 'region_Midwest', 'region_Northeast', 'region_Other', 'region_

## Label encoding

Used to change severity from 1 - 4 to 0 - 3

In [6]:
# This fixes the "Expected [0 1 2 3]" error.
y_encoded, le = label_encode_target(df['Severity'])

--- Label Encoding Component ---
Label mapping:
  1 -> 0
  2 -> 1
  3 -> 2
  4 -> 3


## Train-Test Split

In [7]:
#Uses SMOTE oversampling to balance classes in the training set, then splits into train/test sets
X_train, X_test, y_train, y_test = split_data(X, y_encoded, test_size=0.2, random_state=42)

--- Data Split Component ---
Training set: 400000 samples
Test set: 100000 samples

Training class distribution:
 Class 0: 3487 samples (0.9%)
 Class 1: 318668 samples (79.7%)
 Class 2: 67250 samples (16.8%)
 Class 3: 10595 samples (2.6%)


## Class Weights

In [8]:
classes = np.unique(y_train)
weights = compute_class_weight('balanced', classes=classes, y=y_train)

class_weights = dict(zip(classes, weights))

for k,v in class_weights.items():    
    print(f"{k}: {v:.2f}")

0: 28.68
1: 0.31
2: 1.49
3: 9.44


## Feature Scaling

In [9]:
# Scales features using StandardScaler
X_train_scaled, X_test_scaled, scaler, features = scale_features(X_train, X_test)

--- Scaling Component ---
Features scaled successfully!
Scaler fitted on 113 features.


## SMOTE 

(Only on the scaled training data) 

In [ ]:
# SMOTE 
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)


In [22]:
print(f"After SMOTE, training set class distribution: {pd.Series(y_train_res).value_counts().to_dict()}")

After SMOTE, training set class distribution: {1: 318668, 2: 318668, 3: 318668, 0: 318668}


## Baseline Model

Linear Regression to establish a baseline performance.

In [11]:
#Create a LinearRegression() model instance
baseline_results, baseline_trained, baseline_preds = run_linear_baseline(
    X_train_scaled, 
    X_test_scaled, 
    y_train, 
    y_test, 
    target_stats['range'], 
    target_stats['std']
)


BASELINE MODEL: Linear Regression
Train R²:  0.4598
Test R²:   0.4520
Test RMSE: 0.36
Test MAE:  0.23

--- RMSE in Context ---
RMSE as % of target range: 12.0%
RMSE as % of target std:   74.0%


## Store models results for comparison

In [12]:
# Store all results for comparison
all_results = [baseline_results]

# Dictionary to store trained models
trained_models = {
    'Linear Regression (Baseline)': baseline_trained
}

# Ridge Regression

Ridge adds L2 regularization to prevent overfitting by penalizing large coefficients.

In [13]:
# Run the component
ridge_results, trained_ridge, ridge_preds = run_ridge_model(
    X_train_scaled,  # Original scaled features
    X_test_scaled,   # Original scaled features
    y_train,         # Original labels
    y_test,          # Original labels
    alpha=100
)

# Store the result in your master list
all_results.append(ridge_results)
trained_models['Ridge (Scaled)'] = trained_ridge

------------------------------
Metric          | Value     
------------------------------
Train R²        | 0.4598
Test R²         | 0.4520
R² Gap          | 0.0079
Test RMSE       | 0.36
Manual R²       | 0.4520
------------------------------


## Lasso Regression

In [14]:
#Run the component (Start with a smaller Alpha first!)
# Now call the fixed function
l_res, l_model, l_features = run_lasso_model(
    X_train_scaled, 
    X_test_scaled, 
    y_train, 
    y_test, 
    all_results,     # Pass the dict in
    trained_models,  # Pass the dict in
    alpha=0.001
)

all_results.append(l_res)
trained_models['Lasso'] = l_model

Lasso Regression (Alpha=0.001)
Test R²: 0.4515
Test RMSE: 0.36

Lasso kept 81 of 113 features.


## ElasticNet (L1 + L2 Combined)

In [15]:
e_res, e_model, e_features = run_elastic_net_model(
    X_train_scaled, 
    X_test_scaled, 
    y_train, 
    y_test, 
    all_results, 
    trained_models, 
    alpha=0.001, 
    l1_ratio=0.5
)
all_results.append(e_res)
trained_models['ElasticNet'] = e_model

ElasticNet (L1 + L2 Combined):
  Train R²: 0.4593
  Test R²:  0.4518
  Gap:      0.0075
  Test RMSE: 0.36

ElasticNet kept 95 of 113 features.


## PolynomialFeatures

In [16]:
#Initialize the transformer (degree 2 captures curves and interactions)
#TODO: Uncomment to run the polynomial Lasso model after fixing the feature-dropping issue
#Time Consuming! Use with caution and consider starting with a smaller alpha to reduce the number of features kept.
# p_res, p_model, p_features = run_poly_lasso_model(
#     X_train_scaled, 
#     X_test_scaled, 
#     y_train, 
#     y_test, 
#     all_results, 
#     trained_models, 
#     alpha=0.01  # Prunes the noise
# )

In [17]:
#TODO: Uncomment to run the polynomial Ridge model after fixing the feature-dropping issue
#Time Consuming! Use with caution and consider starting with a smaller alpha to reduce the number of features kept.

# pr_res, pr_model = run_poly_ridge_model(
#     X_train_scaled, 
#     X_test_scaled, 
#     y_train, 
#     y_test, 
#     all_results, 
#     trained_models, 
#     alpha=1.0
# )

## Decision Tree

In [18]:
dt_res, dt_model = run_decision_tree_suite(X_train_scaled, X_test_scaled, y_train, y_test, max_depth=7)

# 2. Store them using your preferred syntax
all_results.append(dt_res)
trained_models['DecisionTree'] = dt_model

Decision Tree (Depth 7) - R²: 0.5056, RMSE: 0.34


In [19]:
#TODO: Uncomment to run the tuned tree model 
#Time Consuming! Use with caution

# Define what you want the computer to test
# my_params = {
#     'max_depth': [3, 5, 7, 10, 15],
#     'min_samples_split': [2, 5, 10],
#     'min_samples_leaf': [1, 2, 4]
# }

# # all the function
# t_res, t_model = run_tuned_tree_model(X_train_scaled, X_test_scaled, y_train, y_test, my_params)

# # Store exactly as you prefer
# all_results.append(t_res)
# trained_models['TunedTree'] = t_model

## Random Forest

In [20]:
# Run the ensemble
rf_res, rf_model_obj = run_random_forest_model(X_train_scaled, X_test_scaled, y_train, y_test, n_estimators=100, max_depth=7)

# 2. Store in your collections
all_results.append(rf_res)
trained_models['Random Forest'] = rf_model_obj

Random Forest - Test R²: 0.5105, Test RMSE: 0.34


## Gradient Boosting

In [21]:
#Run the boosting process
g_res, g_model = run_gradient_boosting_model(X_train_scaled, X_test_scaled, y_train, y_test)

#Store using your preferred syntax
all_results.append(g_res)
trained_models['Gradient Boosting'] = g_model

KeyboardInterrupt: 

# Model Comparison

## Cross-Validation (More Robust Evaluation)

Cross-validation gives us a more reliable estimate of model performance by testing on multiple different train/test splits.


In [ ]:
# Perform 5-fold cross-validation on top models
# top_models = {
#     'Linear Regression': LinearRegression(),
#     'Ridge': Ridge(alpha=1.0),
#     'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
#     'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
# }

# # 2. Run the Leaderboard
# cv_leaderboard_df = run_cv_leaderboard(X_train_scaled, y_train, top_models)

In [ ]:
# Create comparison DataFrame using your CV results
# results_df = pd.DataFrame(cv_leaderboard_df) 
# results_df = results_df.round(3)

# # Sort by the mean score you calculated in the loop
# results_df = results_df.sort_values('CV Mean R2', ascending=False) 

# print("Model Comparison:")
# print(results_df)

## Visualize model comparison

In [ ]:
# Visualize model comparison
# fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# # R² Comparison
# models = results_df['Model']
# x = np.arange(len(models))
# width = 0.35

# axes[0].bar(x, results_df['CV Mean R2'], width, label='Mean CV R2', color='skyblue', alpha=0.8)
# axes[0].set_ylabel('R² Score')
# axes[0].set_title('Mean CV R² Across Models')
# axes[0].set_xticks(x)
# axes[0].set_xticklabels(models, rotation=45, ha='right')
# axes[0].set_ylim(0, 1.1)
# axes[0].axhline(y=0.7, color='green', linestyle='--', alpha=0.5, label='Good threshold')
# axes[0].legend()

# # RMSE Comparison
# axes[1].bar(x, results_df['CV Std R2'], width, label='CV Std Dev', color='salmon', alpha=0.8)
# axes[1].set_ylabel('Standard Deviation')
# axes[1].set_title('Consistency (Std Dev) Across Models')
# axes[1].set_xticks(x)
# axes[1].set_xticklabels(models, rotation=45, ha='right')
# axes[1].legend()

# plt.tight_layout()
# plt.show()

## Feature Importance & Selection

**Important:** Your final model should use only **4-8 features**. This section helps you identify which features matter most.

In [ ]:
# Use the model you already trained in the previous step
rf_model = trained_models['Random Forest']

# Run the visualization and store the full list
rf_importance_df = plot_feature_importance(
    rf_model, 
    X_train_scaled.columns, 
    model_name="Random Forest"
)

# Optional: Add the top insights to your results
all_results.append({
    'Model': 'Feature Importance', 
    'Top Feature': rf_importance_df.iloc[0]['Feature']
})

correlations = X_train.corrwith(y_train).abs().sort_values(ascending=False)
print("Absolute Correlations with Target:")
print(correlations)

In [ ]:
SELECTED_FEATURES = [
    'experience_level',
    'company_location_Other',
    'company_location_US',
    'job_title_Data Analyst',
    'job_title_Machine Learning Engineer',
    'job_title_Software Engineer',
    'job_title_Other'
]

X_train_selected = X_train_scaled[SELECTED_FEATURES]
X_test_selected = X_test_scaled[SELECTED_FEATURES]

selected_results = []

# Your loop now has a function to call!
for name, model in [('Linear Regression', LinearRegression()),
                    ('Ridge', Ridge(alpha=1.0)),
                    ('Random Forest', RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)),
                    ('Gradient Boosting', GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42))]:
    
    # This call now works:
    results, trained, _ = evaluate_model(model, X_train_selected, X_test_selected, y_train, y_test, name)
    
    selected_results.append(results)
    print(f"{name} with {len(SELECTED_FEATURES)} features - Test R²: {results['Test R2']:.4f}")

# Final Comparison Table
selected_df = pd.DataFrame(selected_results)

## Best Model Selection

In [ ]:

final_model_obj, performance_stats = run_final_model_deployment(
    X_train_selected, 
    X_test_selected, 
    y_train, 
    y_test, 
    SELECTED_FEATURES, 
    target_stats['range'], 
    target_stats['std']
)

# Store the final champion
trained_models['Final_GradientBoosting'] = final_model_obj
all_results.append({'Model': 'FINAL DEPLOYMENT', 'Stats': performance_stats})